In [1]:
import os

print(os.getcwd())
print(os.listdir("../data/processed"))


c:\Users\shimo\virality-analysis-datascience\notebooks
['feed_#Disability.parquet', 'feed_#UkrainianView.parquet', 'feed_AcademicSky.parquet', 'feed_Blacksky.parquet', 'feed_bookmarks.parquet', 'feed_BookSky.parquet', 'feed_Game Dev.parquet', 'feed_GreenSky.parquet', 'feed_likes_#Disability.parquet', 'feed_likes_#UkrainianView.parquet', 'feed_likes_AcademicSky.parquet', 'feed_likes_Blacksky.parquet', 'feed_likes_BookSky.parquet', 'feed_likes_Game Dev.parquet', 'feed_likes_GreenSky.parquet', 'feed_likes_News.parquet', 'feed_likes_Political Science.parquet', 'feed_likes_Science.parquet', "feed_likes_What's History.parquet", 'feed_News.parquet', 'feed_Political Science.parquet', 'feed_Science.parquet', "feed_What's History.parquet", 'followers.parquet', 'interactions.parquet', 'posts_sample_small.parquet', 'quotes.parquet', 'replies.parquet', 'reposts.parquet']


In [2]:
import pandas as pd

posts = pd.read_parquet("../data/processed/posts_sample_small.parquet")
interactions = pd.read_parquet("../data/processed/interactions.parquet")
followers = pd.read_parquet("../data/processed/followers.parquet")
reposts = pd.read_parquet("../data/processed/reposts.parquet")
replies = pd.read_parquet("../data/processed/replies.parquet")
print("posts columns:", posts.columns.tolist())
print("reposts columns:", reposts.columns.tolist())
print("replies columns:", replies.columns.tolist())
print("interactions columns:", interactions.columns.tolist())
print(f"Posts: {posts.shape}")
print(f"Interactions: {interactions.shape}")
print(f"Reposts: {reposts.shape}")
print(f"Replies: {replies.shape}")
print(f"Followers: {followers.shape}")

posts columns: ['post_id', 'user_id', 'text', 'langs', 'reply_to', 'thread_root', 'repost_from', 'reposted_author', 'reply_count', 'repost_count', 'created_at']
reposts columns: ['user_id', 'reposted_author', 'date']
replies columns: ['user_id', 'replied_author', 'date']
interactions columns: ['user_id', 'replied_author', 'thread_root_author', 'reposted_author', 'quoted_author', 'date']
Posts: (100000, 11)
Interactions: (152728104, 6)
Reposts: (63438069, 3)
Replies: (87550414, 3)
Followers: (144581603, 2)


In [3]:
posts["created_at"] = pd.to_datetime(posts["created_at"], errors="coerce")
print(posts["created_at"].dtype)
posts = posts.rename(columns={"user_id": "author_id"})
print(posts.columns.tolist())

datetime64[ns]
['post_id', 'author_id', 'text', 'langs', 'reply_to', 'thread_root', 'repost_from', 'reposted_author', 'reply_count', 'repost_count', 'created_at']


In [ ]:
follower_counts = (
    followers
    .groupby("followed_id")
    .size()
    .reset_index(name="follower_count")
    .rename(columns={"followed_id": "author_id"})
)

print(follower_counts.shape)
follower_counts.head()

In [ ]:
original_posts = posts[
    posts["reply_to"].isna() & posts["repost_from"].isna()
].copy()

repost_events = posts[
    posts["repost_from"].notna()
].copy()

reply_events = posts[
    posts["reply_to"].notna()
].copy()

print("original_posts:", original_posts.shape) ## candidate posts that can go viral
print("repost_events:", repost_events.shape) ## rows representing repost actions
print("reply_events:", reply_events.shape) ## rows representing reply actions


In [ ]:
original_posts["text_length"] = original_posts["text"].fillna("").str.len()
original_posts["post_hour"] = original_posts["created_at"].dt.hour
original_posts["post_dayofweek"] = original_posts["created_at"].dt.dayofweek

original_posts.head()

In [ ]:
if "repost_count" not in original_posts.columns:
    raise ValueError("repost_count column not found in posts file.")

viral_threshold = original_posts["repost_count"].quantile(0.95)

original_posts["is_viral"] = (
    original_posts["repost_count"] >= viral_threshold
).astype(int)

print("Virality threshold (top 5%):", viral_threshold)
print(original_posts["is_viral"].value_counts())

## defines viral posts as those in the top 5% of repost counts, and creates a binary target variable 'is_viral' for modeling.

In [ ]:
## allows us to tie reposts/replies back to the original post they act on
original_lookup = original_posts[["post_id", "author_id", "created_at"]].copy()
original_lookup = original_lookup.rename(columns={
    "post_id": "target_post_id",
    "author_id": "target_author_id",
    "created_at": "target_created_at"
})

original_lookup.head()


In [ ]:
EARLY_MINUTES = 10  # try 30, 60, 180 later

repost_events = repost_events.rename(columns={
    "author_id": "reposter_id",
    "repost_from": "target_post_id",
    "created_at": "event_time"
})

repost_events = repost_events.merge(
    original_lookup,
    on="target_post_id",
    how="inner"
)

repost_events["minutes_since_post"] = (
    (repost_events["event_time"] - repost_events["target_created_at"]).dt.total_seconds() / 60
)

repost_events = repost_events[
    repost_events["minutes_since_post"].notna() &
    (repost_events["minutes_since_post"] >= 0)
].copy()

print(repost_events.shape)
repost_events.head()

In [ ]:
reposter_followers = follower_counts.rename(columns={"author_id": "reposter_id"})

repost_events = repost_events.merge(
    reposter_followers,
    on="reposter_id",
    how="left"
)

repost_events["follower_count"] = repost_events["follower_count"].fillna(0)
repost_events.head()

In [ ]:
early_reposts = repost_events[
    repost_events["minutes_since_post"] <= EARLY_MINUTES
].copy()

repost_features = early_reposts.groupby("target_post_id").agg(
    early_repost_count=("reposter_id", "count"),
    unique_early_reposters=("reposter_id", "nunique"),
    avg_reposter_followers=("follower_count", "mean"),
    max_reposter_followers=("follower_count", "max"),
    time_to_first_repost_min=("minutes_since_post", "min")
).reset_index()

print(repost_features.shape)
repost_features.head()

In [ ]:
reply_target_col = "thread_root" if "thread_root" in reply_events.columns else "reply_to"

reply_events = reply_events.rename(columns={
    "author_id": "replier_id",
    "created_at": "event_time"
})

reply_events["target_post_id"] = reply_events[reply_target_col]

reply_events = reply_events.merge(
    original_lookup,
    on="target_post_id",
    how="inner"
)

reply_events["minutes_since_post"] = (
    (reply_events["event_time"] - reply_events["target_created_at"]).dt.total_seconds() / 60
)

reply_events = reply_events[
    reply_events["minutes_since_post"].notna() &
    (reply_events["minutes_since_post"] >= 0)
].copy()

print(reply_events.shape)
reply_events.head()

In [ ]:
early_replies = reply_events[
    reply_events["minutes_since_post"] <= EARLY_MINUTES
].copy()

reply_features = early_replies.groupby("target_post_id").agg(
    early_reply_count=("replier_id", "count"),
    unique_early_repliers=("replier_id", "nunique"),
    time_to_first_reply_min=("minutes_since_post", "min")
).reset_index()

print(reply_features.shape)
reply_features.head()

In [ ]:
author_followers = follower_counts.rename(columns={"author_id": "author_id"})

feature_table = original_posts[[
    "post_id", "author_id", "created_at", "text_length",
    "post_hour", "post_dayofweek", "repost_count", "reply_count", "is_viral"
]].copy()

feature_table = feature_table.merge(
    author_followers,
    on="author_id",
    how="left"
)

feature_table = feature_table.rename(columns={"follower_count": "author_follower_count"})
feature_table.head()

In [ ]:
feature_table = feature_table.merge(
    repost_features,
    left_on="post_id",
    right_on="target_post_id",
    how="left"
).drop(columns=["target_post_id"], errors="ignore")

feature_table = feature_table.merge(
    reply_features,
    left_on="post_id",
    right_on="target_post_id",
    how="left"
).drop(columns=["target_post_id"], errors="ignore")

feature_table.head()

In [ ]:
fill_zero_cols = [
    "author_follower_count",
    "early_repost_count",
    "unique_early_reposters",
    "avg_reposter_followers",
    "max_reposter_followers",
    "early_reply_count",
    "unique_early_repliers"
]

for col in fill_zero_cols:
    if col in feature_table.columns:
        feature_table[col] = feature_table[col].fillna(0)

fill_minus1_cols = [
    "time_to_first_repost_min",
    "time_to_first_reply_min"
]

for col in fill_minus1_cols:
    if col in feature_table.columns:
        feature_table[col] = feature_table[col].fillna(-1)

feature_table.head()

In [ ]:
import numpy as np
feature_table["early_reply_to_repost_ratio"] = np.where(
    feature_table["early_repost_count"] > 0,
    feature_table["early_reply_count"] / feature_table["early_repost_count"],
    0
)

feature_table["has_early_repost"] = (feature_table["early_repost_count"] > 0).astype(int)
feature_table["has_early_reply"] = (feature_table["early_reply_count"] > 0).astype(int)

feature_table["log_author_follower_count"] = np.log1p(feature_table["author_follower_count"])
feature_table["log_text_length"] = np.log1p(feature_table["text_length"])

feature_table.head()

In [ ]:
print(feature_table.shape)
print(feature_table.columns.tolist())
print(feature_table["is_viral"].value_counts())
feature_table.head()